# 01 - ECG Biosignal Exploration & Spectral Analysis
## MedAgent: Multimodal Clinical Decision Support System

This notebook demonstrates the exploratory data analysis (EDA) of continuous ECG signals from the **MIT-BIH Arrhythmia Database**, inspecting waveform morphology, sampling rates, annotation markers, and frequency spectrum characteristics.


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import wfdb

from src.config.paths import DATA_RAW_ECG_DIR
from src.data.preprocessing.segmentation import AAMI_CLASS_NAMES


### 1. Load Raw MIT-BIH Record using `wfdb`


In [ ]:
# Select a representative MIT-BIH record
record_path = DATA_RAW_ECG_DIR / "mit-bih-arrhythmia-database-1.0.0" / "100"

print(f"Loading record from: {record_path}")
record = wfdb.rdrecord(str(record_path))
annotation = wfdb.rdann(str(record_path), 'atr')

print(f"Record Name: {record.record_name}")
print(f"Sampling Frequency: {record.fs} Hz")
print(f"Signal Length: {record.sig_len} samples ({record.sig_len / record.fs / 60:.2f} minutes)")
print(f"Channels: {record.sig_name} ({record.units})")
print(f"Total Annotations (R-peaks): {len(annotation.sample)}")


### 2. Plot Raw ECG Waveforms with Clinical Annotation Markers


In [ ]:
# Extract first 10 seconds of Lead MLII
fs = record.fs
duration_sec = 10
n_samples = duration_sec * fs
time_axis = np.arange(n_samples) / fs

signal_segment = record.p_signal[:n_samples, 0]  # Channel 0: MLII

# Get annotations within this 10-second window
ann_mask = annotation.sample < n_samples
ann_samples = annotation.sample[ann_mask]
ann_symbols = [annotation.symbol[i] for i, m in enumerate(ann_mask) if m]

plt.figure(figsize=(15, 5))
plt.plot(time_axis, signal_segment, label="Lead MLII (Raw)", color="#1f77b4", lw=1.2)
plt.scatter(ann_samples / fs, signal_segment[ann_samples], color="#d62728", s=60, zorder=5, label="R-Peak Annotations")

for s_idx, sym in zip(ann_samples, ann_symbols):
    plt.annotate(sym, (s_idx / fs, signal_segment[s_idx] + 0.15),
                 textcoords="offset points", xytext=(0, 5), ha="center", fontweight="bold")

plt.title("MIT-BIH Record 100 - Raw ECG Waveform & R-Peak Annotations", fontsize=14)
plt.xlabel("Time (seconds)", fontsize=12)
plt.ylabel("Voltage (mV)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


### 3. Fast Fourier Transform (FFT) & Power Spectral Density (PSD)
Evaluating the physiological spectrum ($0.5	ext{--}45	ext{ Hz}$), baseline wander ($<0.5	ext{ Hz}$), and powerline interference ($60	ext{ Hz}$).


In [ ]:
from scipy.fft import rfft, rfftfreq

# Compute FFT on 1-minute segment
one_min_signal = record.p_signal[:fs * 60, 0]
N = len(one_min_signal)
yf = np.abs(rfft(one_min_signal))
xf = rfftfreq(N, 1 / fs)

plt.figure(figsize=(14, 4))
plt.plot(xf, yf, color="#2ca02c", lw=1.2)
plt.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Lower Bandpass (0.5 Hz)")
plt.axvline(45.0, color="gray", linestyle="--", alpha=0.7, label="Upper Bandpass (45.0 Hz)")
plt.axvline(60.0, color="red", linestyle=":", alpha=0.8, label="Powerline Noise (60 Hz)")
plt.xlim(0, 100)
plt.title("Power Spectral Density of Raw ECG Signal", fontsize=14)
plt.xlabel("Frequency (Hz)", fontsize=12)
plt.ylabel("Magnitude", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
